In [4]:
# Import required libraries
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import joblib
import shap
import sys
assert (
    sys.version_info.major == 3 and sys.version_info.minor == 10
), "Please ensure that you are on Python 3.10."

# Load a sample of the data and the models
X_train = pd.read_csv("data/X_train.csv").sample(500, random_state=42)
X_test = pd.read_csv("data/X_test.csv").sample(500, random_state=42)
y_train = pd.read_csv("data/y_train.csv")["nextmonth__home_decor"].sample(500, random_state=42)
y_test = pd.read_csv("data/y_test.csv")["nextmonth__home_decor"].sample(500, random_state=42)
model = joblib.load("data/model.pkl")
knn_model = joblib.load("data/knn_model.pkl")

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator KNeighborsRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitat

In [5]:
# Load data
X_train = pd.read_csv("data/X_train.csv").sample(500, random_state=42)
X_test = pd.read_csv("data/X_test.csv").sample(500, random_state=42)
y_train = pd.read_csv("data/y_train.csv")["nextmonth__home_decor"].sample(500, random_state=42)
y_test = pd.read_csv("data/y_test.csv")["nextmonth__home_decor"].sample(500, random_state=42)

In [6]:
# Load models
model = joblib.load("data/model.pkl")
knn_model = joblib.load("data/knn_model.pkl")

Trying to unpickle estimator DecisionTreeRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator RandomForestRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator KNeighborsRegressor from version 1.6.1 when using version 1.3.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [7]:
# SHAP for Random Forest
rf_explainer = shap.TreeExplainer(model)
rf_shap_values = rf_explainer.shap_values(X_test)
rf_importance = np.abs(rf_shap_values).mean(axis=0)

In [8]:
rf_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": rf_importance
}).sort_values(by="Importance", ascending=False)

top_rf_feats = rf_importance_df.head(5)

In [9]:
# SHAP for k-NN
knn_explainer = shap.KernelExplainer(knn_model.predict, shap.kmeans(X_test, 5))
knn_shap_values = knn_explainer.shap_values(X_test.sample(50, random_state=42))
knn_importance = np.abs(knn_shap_values).mean(axis=0)

100%|██████████| 50/50 [00:17<00:00,  2.85it/s]


In [10]:
knn_importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": knn_importance
}).sort_values(by="Importance", ascending=False)

top_knn_feats = knn_importance_df.head(5)

In [11]:
# Compute consistency
consistency_score = round(
    cosine_similarity([rf_importance], [knn_importance])[0][0], 2
)
print("Consistency between SHAP values:", consistency_score)

Consistency between SHAP values: 0.89
